# Question Answering con DistilBERT — Teoria e codice commentato

Questo notebook riprende `3_QuestionAnswering.ipynb` e lo arricchisce con teoria introduttiva, una roadmap e commenti estesi al codice.

---

## 1. Teoria introduttiva

### 1.1 Cos'è il Question Answering (QA) estrattivo
Il **question answering estrattivo** è il compito di individuare, all'interno di un testo di riferimento (**context**), la porzione esatta di testo (**span**) che risponde a una **domanda**. A differenza della summarization (dove il modello *genera* nuovo testo), qui il modello **seleziona** un frammento già presente nel context: non inventa parole nuove, indica solo dove inizia e dove finisce la risposta.

In pratica il modello impara a predire, per ogni token del context:
- la probabilità che quel token sia l'**inizio** della risposta;
- la probabilità che quel token sia la **fine** della risposta.

La risposta finale è la sotto-sequenza di token tra la posizione di inizio e quella di fine con probabilità (combinata) più alta.

### 1.2 BERT e l'architettura encoder-only
**BERT (Bidirectional Encoder Representations from Transformers)** usa solo lo **stack di encoder** del Transformer (a differenza di GPT-2, decoder-only, e di T5, encoder-decoder). Non essendo autoregressivo, ogni token può "guardare" **sia a sinistra che a destra** nella frase: la rappresentazione che il modello costruisce per ogni parola è quindi **bidirezionale**, il che lo rende molto efficace per compiti di *comprensione* del testo (come il QA), più che di *generazione*.

### 1.3 Pretraining di BERT
BERT è pre-addestrato con due obiettivi self-supervised (si vedano anche gli esempi nel notebook sulla summarization):
- **Masked Language Modeling (MLM)**: si maschera circa il 15% dei token in input e il modello deve predire le parole mascherate usando il contesto sia a sinistra che a destra.
- **Next Sentence Prediction (NSP)**: il modello riceve due frasi e deve predire se la seconda segue effettivamente la prima nel testo originale.

Nessuno di questi due obiettivi richiede etichette umane: sono ottenuti automaticamente a partire da testo grezzo, esattamente come il next-token prediction di GPT-2 o lo span corruption di T5.

### 1.4 DistilBERT: distillazione della conoscenza
Il modello usato in questo notebook è **DistilBERT**, una versione **compressa** di BERT ottenuta tramite **knowledge distillation**: un modello più piccolo (lo *studente*) viene addestrato a imitare il comportamento di un modello più grande già addestrato (il *maestro*, BERT). Il risultato è un modello circa il 40% più piccolo e significativamente più veloce, che mantiene comunque la maggior parte delle capacità del modello originale — un buon compromesso quando le risorse di calcolo sono limitate (nota infatti il log `Device set to use cpu`: il modello è abbastanza leggero da girare bene anche su CPU).

### 1.5 Fine-tuning per il QA
Il modello usato, `distilbert-base-cased-distilled-squad`, non è solo pre-addestrato in modo generico: è stato anche **fine-tuned** sul dataset **SQuAD** (Stanford Question Answering Dataset), che contiene migliaia di coppie (context, question, answer) etichettate da esseri umani. Questo è diverso dagli altri due notebook: lì si usava il modello pre-addestrato "puro" (GPT-2) o pre-addestrato per il task generico (T5 text-to-text); qui invece il modello ha già visto esempi supervisionati specifici per il QA estrattivo, il che spiega la qualità molto alta delle risposte anche su esempi semplici.

### 1.6 La pipeline di 🤗 Transformers
A differenza dei notebook precedenti (dove si gestivano esplicitamente tokenizer e modello), qui si usa `pipeline('question-answering', ...)`: un'utility di alto livello che incapsula automaticamente tokenizzazione, forward pass del modello, ed estrazione/decodifica della risposta a partire dai logit di inizio/fine. È più comodo per task "standard" come questo, a scapito di un minor controllo sui dettagli (che invece si vedevano nei notebook su GPT-2 e T5).

## 2. Dove stiamo andando (roadmap del notebook)

1. **Setup ambiente**: installiamo `tf-keras` (dipendenza di `transformers`).
2. **Import**: `torch` e la funzione `pipeline` di `transformers`.
3. **Creazione della pipeline di QA**, che carica automaticamente sia il modello (`distilbert-base-cased-distilled-squad`) sia il tokenizer corrispondente.
4. **Definizione della classe `LLMApplications`** con il metodo `question_answering`, che chiama la pipeline e ne estrae la risposta testuale.
5. **Primo esempio**: una domanda semplice su una frase molto corta (definizione di "transformers").
6. **Secondo esempio**: domanda su un fatto storico (chi ha progettato la Torre Eiffel).
7. **Terzo esempio**: domanda che richiede di individuare una motivazione/scopo (perché è stata costruita la Grande Muraglia).

### 2.1 Setup dell'ambiente

In [1]:
# Installa tf-keras: dipendenza richiesta da transformers per compatibilità
# con l'API Keras legacy (il modello verrà comunque eseguito con backend PyTorch).
! pip install tf-keras


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 2.2 Import

In [2]:
# torch: framework usato sotto il cofano dalla pipeline per l'esecuzione del modello.
import torch

# pipeline: API di alto livello di transformers che incapsula
# tokenizzazione + modello + post-processing per un task standard
# (qui 'question-answering'), senza doverli gestire manualmente come
# fatto nei notebook precedenti con GPT-2 e T5.
from transformers import pipeline

### 2.3 Creazione della pipeline di Question Answering

`distilbert-base-cased-distilled-squad` è già fine-tuned su SQuAD: la pipeline scarica sia i pesi del modello sia la configurazione del tokenizer associato, pronti per l'inferenza.

In [3]:
# Modello per domande e risposte (DistilBERT)
# 'question-answering' seleziona automaticamente la logica di
# pre/post-processing corretta per il QA estrattivo (calcolo degli span
# di inizio/fine risposta a partire dai logit del modello).
qa_pipeline = pipeline('question-answering', model='distilbert-base-cased-distilled-squad', tokenizer='distilbert-base-cased-distilled-squad')

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

c:\Users\A1618apulia\OneDrive - ITS Apulia Digital Maker\Desktop\GENERATIVEAI\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\A1618apulia\.cache\huggingface\hub\models--distilbert-base-cased-distilled-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


KeyError: "Unknown task question-answering, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

### 2.4 La classe `LLMApplications` con il metodo `question_answering`

Stesso pattern architetturale usato nei notebook precedenti (dizionario `self.applications` estensibile), ma qui il metodo è una sottile wrapper attorno alla pipeline, che fa già quasi tutto il lavoro.

In [ ]:
class LLMApplications:
    def __init__(self):
        # Inizializza le applicazioni
        self.applications = {
            'question_answering': self.question_answering
        }

    def question_answering(self, context, question):
        """Rispondi a una domanda utilizzando DistilBERT"""

        # La pipeline si occupa internamente di:
        # 1) tokenizzare context e question insieme,
        # 2) eseguire il forward pass del modello per ottenere i logit
        #    di inizio/fine risposta su ogni token del context,
        # 3) selezionare lo span più probabile e ri-decodificarlo in testo.
        # Il risultato è un dizionario con almeno le chiavi 'answer',
        # 'score' (confidenza) e le posizioni 'start'/'end' nel context.
        result = qa_pipeline(question=question, context=context)

        # Estrae solo il testo della risposta (ignoriamo qui score/start/end,
        # ma sono disponibili in 'result' per eventuali usi più avanzati,
        # es. per scartare risposte con score troppo basso).
        return result['answer']

### 2.5 Primo esempio: domanda su una frase molto corta

In [ ]:
llm_apps = LLMApplications()

#  Question Answering
print("QUESTION ANSWERING:")
context = "Transformers are neural networks that use attention mechanisms."
question = "What are transformers?"

# La risposta sarà uno span estratto letteralmente dal context, non una
# frase generata da zero: qui ci aspettiamo qualcosa come "neural networks".
answer = llm_apps.question_answering(context, question)
print(f"   Context: {context}")
print(f"   Question: {question}")
print(f"   Answer: {answer}")
print()

### 2.6 Secondo esempio: fatto storico (Torre Eiffel)

Domanda "who", tipicamente più semplice per un modello fine-tuned su SQuAD: la risposta è quasi sempre un'entità nominata (nome proprio) chiaramente identificabile nel testo.

In [ ]:
# Another Question Answering Example
print("QUESTION ANSWERING:")
context = "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It was named after the engineer Gustave Eiffel, whose company designed and built the tower."
question = "Who designed the Eiffel Tower?"
answer = llm_apps.question_answering(context, question)
print(f"   Context: {context}")
print(f"   Question: {question}")
print(f"   Answer: {answer}")
print()

### 2.7 Terzo esempio: domanda sullo scopo/motivazione (Grande Muraglia)

Domanda "what was the purpose", più impegnativa di una semplice "who": la risposta non è un'entità nominata isolata, ma una porzione di frase che esprime uno scopo ("to protect northern China"). Utile per osservare che il modello sa individuare span più lunghi e semanticamente più complessi, non solo singole parole.

In [ ]:
# Another Question Answering Example
print("QUESTION ANSWERING:")
context = "The Great Wall of China is a series of fortifications made of various materials like stone, brick, tamped earth, and others. It was built to protect northern China from invasions and raids by various nomadic groups of the Eurasian Steppe."
question = "What was the purpose of the Great Wall of China?"
answer = llm_apps.question_answering(context, question)
print(f"   Context: {context}")
print(f"   Question: {question}")
print(f"   Answer: {answer}")
print()

## 3. Cosa osservare nei risultati

- Le risposte sono sempre **estratte letteralmente** dal context: il modello non può "inventare" parole che non siano già presenti nel testo di input. Se la risposta corretta non è contenuta nel context, il modello restituirà comunque lo span che considera più probabile, ma sarà per forza scorretto o parziale.
- A differenza dei due notebook precedenti (GPT-2 e T5), qui non c'è generazione libera: il compito è **selezione**, non produzione di nuovo testo. Per questo il risultato è deterministico (nessun `do_sample`), a parità di context/question.
- La qualità molto alta delle risposte, anche su esempi semplici, è dovuta al **fine-tuning su SQuAD**: il modello ha già visto migliaia di esempi di coppie (context, question, answer) etichettate da persone, non solo il pretraining self-supervised generico (MLM + NSP).
- Il log `Device set to use cpu` conferma che, grazie alla distillazione, il modello è abbastanza leggero da essere eseguito comodamente anche senza GPU.
- Il dizionario `result` restituito dalla pipeline contiene anche `score` (confidenza della risposta) e le posizioni `start`/`end` nel context: in un'applicazione reale sarebbe buona pratica controllare `score` per scartare risposte poco affidabili, invece di restituire sempre e comunque `result['answer']`.